# ISLES'24 NCCT-only five-fold OOF evaluation

This notebook evaluates the completed 149-case out-of-fold NCCT nnU-Net baseline. **GPU is not required.**

It verifies the five deterministic validation folds, checks that every public training case appears exactly once OOF, checks NIfTI geometry, computes the four public ISLES'24 metrics (Dice, AVD, lesion-wise F1, absolute lesion-count difference), cross-checks Dice against nnU-Net's validation summaries, and writes persistent CSV/JSON outputs to Google Drive.

Important: the public ISLES'24 notebook states that official challenge evaluation masks lesions with an NCCT-derived brain mask. The organizer-compatible brain-mask generation step is not published in the public evaluator. Unless such masks are supplied, this notebook labels the outputs as **local OOF metrics using the public official metric implementation, without organizer brain-mask postprocessing**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Paths and repository


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

ROOT = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
NNUNET_PREPROCESSED = ROOT / 'nnUNet_preprocessed'
NNUNET_RESULTS = ROOT / 'nnUNet_results'
OUTPUT = ROOT / 'results' / 'ncct_oof_evaluation'
REPO_ZIP = Path('/content/HybridStrokeSeg-main.zip')
REPO = Path('/content/HybridStrokeSeg-Reproduction-main')

# Leave this as None unless we later obtain organizer-compatible NCCT-derived brain masks.
BRAIN_MASK_ROOT = None

OUTPUT.mkdir(parents=True, exist_ok=True)

def run(cmd, *, cwd=None):
    cmd = [str(x) for x in cmd]
    print('$', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True, cwd=cwd)

if REPO.exists():
    shutil.rmtree(REPO)
if REPO_ZIP.exists():
    REPO_ZIP.unlink()

run([
    'curl', '-fL', '--retry', '20', '--retry-delay', '3', '--retry-all-errors',
    'https://github.com/NTinkicht/HybridStrokeSeg-Reproduction/archive/refs/heads/main.zip',
    '-o', REPO_ZIP,
])
run(['unzip', '-q', REPO_ZIP, '-d', '/content'])
run(['python', '-m', 'pip', 'install', '-q', '-e', '.[evaluation]'], cwd=REPO)


## Run verified OOF evaluation


In [ ]:
cmd = [
    'python', 'scripts/evaluate_isles2024_nnunet_oof.py',
    '--dataset-id', '504',
    '--configuration', '3d_fullres',
    '--preprocessed-root', NNUNET_PREPROCESSED,
    '--results-root', NNUNET_RESULTS,
    '--output-dir', OUTPUT,
    '--bootstrap-repetitions', '10000',
    '--bootstrap-seed', '2026',
]
if BRAIN_MASK_ROOT is not None:
    cmd += ['--brain-mask-root', BRAIN_MASK_ROOT]

run(cmd, cwd=REPO)


## Results summary


In [ ]:
summary_path = OUTPUT / 'isles2024_ncct_oof_summary.json'
manifest_path = OUTPUT / 'isles2024_ncct_oof_manifest.json'
csv_path = OUTPUT / 'isles2024_ncct_oof_case_metrics.csv'

summary = json.loads(summary_path.read_text())
print(json.dumps({
    'n_oof_cases': summary['n_oof_cases'],
    'unique_oof_cases': summary['unique_oof_cases'],
    'brain_mask_applied': summary['brain_mask_applied'],
    'metrics': summary['metrics'],
    'nnunet_dice_crosscheck': summary['nnunet_dice_crosscheck'],
}, indent=2))
print('\nCase CSV:', csv_path)
print('Summary:', summary_path)
print('Manifest:', manifest_path)
